In [2]:
import pymupdf4llm
import re
import os
from langchain_text_splitters import MarkdownHeaderTextSplitter, JSFrameworkTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_chroma import Chroma
from dotenv import load_dotenv
import json
load_dotenv()

True

In [3]:
file_path = "Attention.pdf"
markdown = pymupdf4llm.to_markdown(file_path)

rapidocr_api using backend: rapidocr

=== Document parser messages ===
Using RapidOCR for OCR processing.
OCR on page.number=2/3.
OCR on page.number=3/4.


In [4]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4")
]

splitter = MarkdownHeaderTextSplitter(headers_to_split_on = headers_to_split_on)
markdown_chunks = splitter.split_text(markdown)

In [5]:
def additional_recursive_chunker(markdown_chunks : list[Document], chunk_size = 500, chunk_overlap = 100):

    good_chunk = []
    chunk_count = 0
    recursive_text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, 
                                                             chunk_overlap = chunk_overlap, 
                                                             separators=["\n\n", "\n", ". ", " ", ""])
    id = 1
    for chunk in markdown_chunks:
        if(len(chunk.page_content) > chunk_size):
            meta_data = chunk.metadata
            chunks = recursive_text_splitter.split_documents([chunk])
            for i in chunks:
                i.metadata['id'] = id
                chunk_count = chunk_count + 1
                i.metadata["chunk_id"] = chunk_count
                good_chunk.append(i)
                id += 1
            chunk_count = 0
        else:
            chunk.metadata['id'] = id
            id += 1
            good_chunk.append(chunk)
    return good_chunk

In [6]:
# this is the final chunk and metadata which is to be stores and embedded
final_chunks = additional_recursive_chunker(markdown_chunks)

In [7]:
final_chunks[27]

Document(metadata={'Header 1': '**Attention Is All You Need**', 'Header 2': '**3 Model Architecture**', 'Header 3': 'Ouput Probabilities', 'Header 4': '**3.2.1 Scaled Dot-Product Attention**', 'id': 28}, page_content='We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of queries and keys of dimension _dk_ , and values of dimension _dv_ . We compute the dot products of the  \n3')

In [8]:
embedding_model = OllamaEmbeddings(model="llama3.2")

In [9]:
# embed each of the query and then store them in the pgvector
vector_store = Chroma(
    collection_name= "my_docs",
    embedding_function=embedding_model,
    persist_directory="./vectors"
)

In [ ]:
ids = vector_store.add_documents(
    documents = final_chunks,)

In [10]:
filter_condition = {}
search_kwargs = {}

In [11]:
if len(filter_condition) == 0:
    search_kwargs = {
        'k' : 5,
        "fetch_k" : 10,
        "lambda_mult" : 0.7,
    }
else:
    search_kwargs = {
        'k' : 5,
        "fetch_k" : 10,
        "lambda_mult" : 0.7,
        "filter" : {
            **filter_condition
        }
    },

In [12]:
search_kwargs

{'k': 5, 'fetch_k': 10, 'lambda_mult': 0.7}

In [13]:
retriver = vector_store.as_retriever(
    search_type = "mmr",
    search_kwargs = search_kwargs
)

### Evaluation of the rag system without hybrid search and reranking

#### Evaluation LLMs

In [14]:
from openai import OpenAI
from groq import Groq
from ragas.llms import llm_factory

OllamaClient = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434"
)

GroqClient = Groq(
    api_key=os.getenv("api_key")
)

GroqEvaluation = llm_factory(
    model="llama-3.3-70b-versatile",
    client=GroqClient
)

OllamaEvaluation = llm_factory(
    "ollama",
    provider="openai",
    client=OllamaClient
)

c:\Users\Nitro\Desktop\productionReadyRAGapp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Nitro\Desktop\productionReadyRAGapp\venv\Lib\site-packages\ragas\llms\base.py:556: UserWarning: Client should be an instance of openai.OpenAI or openai.AsyncOpenAI. Unexpected behavior may occur with other client types.
  return instructor.from_openai(client, mode=instructor.Mode.JSON)


#### RAG LLMs

In [15]:
OllamaLLM = ChatOllama(model="llama3.2")
GroqLLM = ChatGroq(
    api_key=os.getenv("api_key"),
    model="llama-3.3-70b-versatile"
)

#### Metrics

In [16]:
from ragas.metrics.collections import ( 
    ContextPrecision,
    ContextRecall, 
    ContextRelevance, 
    Faithfulness, 
    AnswerCorrectness, 
    AnswerRelevancy )
from ragas import evaluate
from datasets import Dataset

In [17]:
faithulness = Faithfulness(llm = OllamaEvaluation)
contextRecall = ContextRecall(llm = OllamaEvaluation)
contextPrecision = ContextPrecision(llm = OllamaEvaluation)
contextRelevance = ContextRelevance(llm = OllamaEvaluation)
answerCorrectness = AnswerCorrectness(llm = OllamaEvaluation, embeddings = embedding_model)
# answerRelevancy = AnswerRelevancy(llm = OllamaEvaluation, embeddings= embedding_model) 

In [18]:
# evaluation retriever
def recall_k(total_relevant_chunks, total_relevant_chunks_retirevied):
    # -> how many of relevant chunks were retreived out of all the relevant chunks available
    if total_relevant_chunks == 0:
        return 0.0
    recall = total_relevant_chunks_retirevied/total_relevant_chunks
    return recall

def precision_k(total_retreived_chunks, total_relevant_ones):
    # -> how many of the retrieved chunks were relevant chunks to the the question
    if total_retreived_chunks == 0:
        return 0.0
    precision = total_relevant_ones/total_retreived_chunks
    return precision

# evaluate the ranker (reranker or retriever's ranker)
def rr(rank_of_relevant_chunk):
    # -> the first relevant chunk out of all the retrieved chunk which are returned in order
    rr_value = 1/rank_of_relevant_chunk if rank_of_relevant_chunk != 0 else 0
    return rr_value

def ndcg(llm):
    # -> this metrics check whether the chunks are arranged in an descending order of chunks on 
    # the basis of relevancy
    pass

#### dataset for evaluation
the golden dataset will contain the dataset which will contain:
- user_query
- response
- retreived_contexts
- reference 

In [19]:
golden_dataset = [
    {
        "user_input": "What is the Transformer architecture?",
        "reference": "The Transformer is a neural network architecture based entirely on attention mechanisms, replacing recurrence and convolutions.",
        "retrieved_contexts": [],
        "response": ""
    },
    {
        "user_input": "What is self-attention and how does it work?",
        "reference": "Self-attention is an attention mechanism in which queries, keys, and values come from the same source, specifically the output of the previous layer. It allows different positions in a sequence to be related when computing its representation.",
        "retrieved_contexts": [],
        "response": ""
    },
    {
        "user_input": "What are Query, Key, and Value in the attention mechanism?",
        "reference": "An attention function maps a query and a set of key-value pairs to an output. Queries and keys are used to determine attention scores, while values contribute to the resulting output.",
        "retrieved_contexts": [],
        "response": ""
    },
    {
        "user_input": "Why does the Transformer use multi-head attention?",
        "reference": "Multi-head attention allows the Transformer to jointly attend to information from different representation subspaces at different positions, enabling it to capture different types of relationships simultaneously.",
        "retrieved_contexts": [],
        "response": ""
    },
    {
        "user_input": "What is positional encoding and why is it necessary?",
        "reference": "Positional encoding injects information about the relative or absolute positions of tokens into the model. It is necessary because the Transformer contains neither recurrence nor convolution and therefore needs positional information to understand token order.",
        "retrieved_contexts": [],
        "response": ""
    },
    {
        "user_input": "What is the difference between encoder-decoder attention and masked self-attention?",
        "reference": "Encoder-decoder attention allows every decoder position to attend to all positions in the input sequence through encoder outputs. In contrast, masked self-attention prevents decoder positions from attending to subsequent positions, ensuring that future tokens are not accessed.",
        "retrieved_contexts": [],
        "response": ""
    }
]

In [72]:
relevant_chunks_dataset = [
    {
        "query": "What is the Transformer architecture?",
        "relevant_chunks": [
            {
                "id": 3,
                "section": "Abstract",
                "chunk_id": 1,
                "relevance_score": 3,
                "content": """
The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.
"""
            }
        ]
    },
    {
        "query": "What is self-attention and how is it used in the Transformer?",
        "relevant_chunks": [
            {
                "id": 16,
                "section": "Background",
                "chunk_id": 3,
                "relevance_score": 3,
                "content": """
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations.
"""
            },
            {
                "id": 40,
                "section": "Applications of Attention in our Model",
                "chunk_id": 2,
                "relevance_score": 3,
                "content": """
The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder.
"""
            },
            {
                "id": 41,
                "section": "Applications of Attention in our Model",
                "chunk_id": 3,
                "relevance_score": 2,
                "content": """
Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward information flow in the decoder to preserve the auto-regressive property. We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections.
"""
            }
        ]
    },
    {
        "query": "What is scaled dot-product attention?",
        "relevant_chunks": [
            {
                "id": 27,
                "section": "Attention",
                "chunk_id": 1,
                "relevance_score": 2,
                "content": """
An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key.
"""
            },
            {
                "id": 28,
                "section": "Scaled Dot-Product Attention",
                "chunk_id": 0,
                "relevance_score": 2,
                "content": """
We call our particular attention "Scaled Dot-Product Attention". The input consists of queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the queries with all keys.
"""
            },
            {
                "id": 29,
                "section": "Multi-Head Attention",
                "chunk_id": 1,
                "relevance_score": 3,
                "content": """
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel. The query is multiplied with all keys, divided by √dk, and a softmax function is applied to obtain the weights on the values.
"""
            }
        ]
    },
    {
        "query": "How does multi-head attention work in the Transformer?",
        "relevant_chunks": [
            {
                "id": 34,
                "section": "Multi-Head Attention",
                "chunk_id": 1,
                "relevance_score": 3,
                "content": """
Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions.
"""
            },
            {
                "id": 35,
                "section": "Multi-Head Attention",
                "chunk_id": 2,
                "relevance_score": 2,
                "content": """
These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2.
"""
            },
            {
                "id": 36,
                "section": "Multi-Head Attention",
                "chunk_id": 3,
                "relevance_score": 2,
                "content": """
Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this.
"""
            }
        ]
    },
    {
        "query": "What is positional encoding and why is it needed in the Transformer?",
        "relevant_chunks": [
            {
                "id": 46,
                "section": "Positional Encoding",
                "chunk_id": 1,
                "relevance_score": 3,
                "content": """
Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of the tokens in the sequence.
"""
            },
            {
                "id": 48,
                "section": "Positional Encoding",
                "chunk_id": 3,
                "relevance_score": 2,
                "content": """
The positional encodings have the same dimension d_model as the embeddings, so that the two can be summed. There are many choices of positional encodings, learned and fixed.
"""
            },
            {
                "id": 50,
                "section": "Positional Encoding",
                "chunk_id": 5,
                "relevance_score": 2,
                "content": """
That is, each dimension of the positional encoding corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000 · 2π. We chose this function because we hypothesized it would allow the model to easily learn to attend by relative positions.
"""
            }
        ]
    },
    {
        "query": "How is attention used in the encoder and decoder of the Transformer?",
        "relevant_chunks": [
            {
                "id": 39,
                "section": "Applications of Attention in our Model",
                "chunk_id": 1,
                "relevance_score": 3,
                "content": """
In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence.
"""
            },
            {
                "id": 40,
                "section": "Applications of Attention in our Model",
                "chunk_id": 2,
                "relevance_score": 3,
                "content": """
The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder.
"""
            },
            {
                "id": 41,
                "section": "Applications of Attention in our Model",
                "chunk_id": 3,
                "relevance_score": 3,
                "content": """
Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward information flow in the decoder to preserve the auto-regressive property. We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections.
"""
            }
        ]
    }
]

In [74]:
precision_recall_dataset = []
for question in relevant_chunks_dataset:
    question_dict = {}
    question_dict['query'] = question['query']
    question_dict['relevant_chunk_ids'] = [ chunk['id'] for chunk in question['relevant_chunks']]
    question_dict['relevancy_score'] = [ chunk['relevance_score'] for chunk in question['relevant_chunks']]
    precision_recall_dataset.append(question_dict)

precision_recall_dataset

[{'query': 'What is the Transformer architecture?',
  'relevant_chunk_ids': [3],
  'relevancy_score': [3]},
 {'query': 'What is self-attention and how is it used in the Transformer?',
  'relevant_chunk_ids': [16, 40, 41],
  'relevancy_score': [3, 3, 2]},
 {'query': 'What is scaled dot-product attention?',
  'relevant_chunk_ids': [27, 28, 29],
  'relevancy_score': [2, 2, 3]},
 {'query': 'How does multi-head attention work in the Transformer?',
  'relevant_chunk_ids': [34, 35, 36],
  'relevancy_score': [3, 2, 2]},
 {'query': 'What is positional encoding and why is it needed in the Transformer?',
  'relevant_chunk_ids': [46, 48, 50],
  'relevancy_score': [3, 2, 2]},
 {'query': 'How is attention used in the encoder and decoder of the Transformer?',
  'relevant_chunk_ids': [39, 40, 41],
  'relevancy_score': [3, 3, 3]}]

In [75]:
# retrieve the chunk from the vector store
for key, question in enumerate(relevant_chunks_dataset):
    retrieved_chunks = retriver.invoke(question['query'])
    retrieved_chunks_ids = [ chunk.metadata['id'] for chunk in retrieved_chunks]
    precision_recall_dataset[key]['retrieved_chunk_ids'] = retrieved_chunks_ids

precision_recall_dataset

[{'query': 'What is the Transformer architecture?',
  'relevant_chunk_ids': [3],
  'relevancy_score': [3],
  'retrieved_chunk_ids': [84, 13, 21, 16, 86]},
 {'query': 'What is self-attention and how is it used in the Transformer?',
  'relevant_chunk_ids': [16, 40, 41],
  'relevancy_score': [3, 3, 2],
  'retrieved_chunk_ids': [84, 21, 13, 40, 16]},
 {'query': 'What is scaled dot-product attention?',
  'relevant_chunk_ids': [27, 28, 29],
  'relevancy_score': [2, 2, 3],
  'retrieved_chunk_ids': [84, 40, 32, 27, 29]},
 {'query': 'How does multi-head attention work in the Transformer?',
  'relevant_chunk_ids': [34, 35, 36],
  'relevancy_score': [3, 2, 2],
  'retrieved_chunk_ids': [84, 21, 40, 16, 13]},
 {'query': 'What is positional encoding and why is it needed in the Transformer?',
  'relevant_chunk_ids': [46, 48, 50],
  'relevancy_score': [3, 2, 2],
  'retrieved_chunk_ids': [84, 21, 13, 40, 92]},
 {'query': 'How is attention used in the encoder and decoder of the Transformer?',
  'relevan

#### Precision and Recall

In [76]:
total_precision = []
total_recall = []
for question in precision_recall_dataset:

    # total relevant among retreived
    relevant_among_retrieved = 0
    for i in retrieved_chunks_ids:
        if i in question['relevant_chunk_ids']:
            relevant_among_retrieved += 1

    # precision calculation
    precision_value = precision_k(5, relevant_among_retrieved)
    # recall calcualtion
    recall_value = recall_k(len(question["relevant_chunk_ids"]), relevant_among_retrieved)

    total_precision.append(precision_value)
    total_recall.append(recall_value)


average_precision = sum(total_precision)/5
average_recall = sum(total_recall)/5

print("Precision:", average_precision)
print("Recall:", average_recall)

Precision: 0.16
Recall: 0.26666666666666666


#### MRR and nDGC

In [77]:
# MRR
from functools import reduce
# calculate the rank for each of the query (RR):-
ranks = []
number_of_chunks = 5
for question in precision_recall_dataset:
    status = False
    for rank, retrieved_chunk_id in enumerate(question['retrieved_chunk_ids']):
        if retrieved_chunk_id in question['relevant_chunk_ids']:
            # print(rank+1, retrieved_chunk_id)
            # print(question['relevant_chunk_ids'])
            status = True
            ranks.append(1/(rank+1))
            break
    if(status == False):
        ranks.append(0)
print(ranks)

# calculating MRR now:-
mrr = reduce(lambda acc, current : acc + current, ranks, 0)/number_of_chunks
print("mean reciprocal rank =>", mrr)

[0, 0.25, 0.25, 0, 0, 0.5]
mean reciprocal rank => 0.2


In [106]:
# NDCG
# it is used to check the overall ranking, not only the first chunks as mrr.
# it sees the chunk and then sees if the chunk is ranked above the less relevant or irrelevant chunk.
score = []
retrieved_chunk = []
for question in precision_recall_dataset:
    relevancy = []
    chunk = []
    for id in question['retrieved_chunk_ids']:
        chunk.append(id)
        if id in question['relevant_chunk_ids']:
            index = question['relevant_chunk_ids'].index(id)
            relevancy_chunk_score = question['relevancy_score'][index]
            relevancy.append(relevancy_chunk_score)
        else:
            relevancy.append(0)
    score.append(relevancy)
    retrieved_chunk.append(chunk)
print(score)
print(retrieved_chunk)

[[0, 0, 0, 0, 0], [0, 0, 0, 3, 3], [0, 0, 0, 2, 3], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 3, 0, 0, 0]]
[[84, 13, 21, 16, 86], [84, 21, 13, 40, 16], [84, 40, 32, 27, 29], [84, 21, 40, 16, 13], [84, 21, 13, 40, 92], [21, 40, 84, 27, 29]]


In [ ]:
# calculate DCG:-
for question_chunks_score, relevant_chunks_score in zip(score, retrieved_chunk):
    print(question_chunks_score, relevant_chunks_score)
    break

[0, 0, 0, 0, 0] [84, 13, 21, 16, 86]


In [26]:
eval_dataset = Dataset.from_list(golden_dataset)
eval_dataset

Dataset({
    features: ['user_input', 'reference', 'retrieved_contexts', 'response'],
    num_rows: 6
})

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

### RAG with hybrid search

In [27]:
# we need to perform the retrieval on the basis of the semantic and the keyword and the fuse then perform rrf on them to find the best chunk
# semantic search is the normal search for this


### BM25 as the keyword search

In [28]:
# keyword search is done using the bm25 algorithm

### Combining the chunks from dense and sparse search

In [29]:
# rrf

In [30]:
# reranker

### Generation of the answer from the llm

In [31]:
# generation

### Evaluate the chunks and answer after the changes in the pipeline

In [32]:
# evaluation criterias